In [4]:
import numpy as np
import os
from rdkit import Chem, RDConfig, rdBase
from rdkit.Chem import AllChem, ChemicalFeatures
    
# the dataset can be downloaded from
# https://nmrshiftdb.nmr.uni-koeln.de/portal/js_pane/P-Help
# nmrshiftdb2withsignals.sd

def get_atom_shifts(mol):
    
    molprops = mol.GetPropsAsDict()
    
    atom_shifts = {}
    for key in molprops.keys():
    
        if key.startswith('Spectrum 13C'):
            
            for shift in molprops[key].split('|')[:-1]:
            
                [shift_val, _, shift_idx] = shift.split(';')
                shift_val, shift_idx = float(shift_val), int(shift_idx)
            
                if shift_idx not in atom_shifts: atom_shifts[shift_idx] = []
                atom_shifts[shift_idx].append(shift_val)

    return atom_shifts

def add_mol(mol_dict, mol):

    def _DA(mol):

        D_list, A_list = [], []
        for feat in chem_feature_factory.GetFeaturesForMol(mol):
            if feat.GetFamily() == 'Donor': D_list.append(feat.GetAtomIds()[0])
            if feat.GetFamily() == 'Acceptor': A_list.append(feat.GetAtomIds()[0])
        
        return D_list, A_list

    def _chirality(atom):

        if atom.HasProp('Chirality'):
            #assert atom.GetProp('Chirality') in ['Tet_CW', 'Tet_CCW']
            c_list = [(atom.GetProp('Chirality') == 'Tet_CW'), (atom.GetProp('Chirality') == 'Tet_CCW')] 
        else:
            c_list = [0, 0]

        return c_list

    def _stereochemistry(bond):

        if bond.HasProp('Stereochemistry'):
            #assert bond.GetProp('Stereochemistry') in ['Bond_Cis', 'Bond_Trans']
            s_list = [(bond.GetProp('Stereochemistry') == 'Bond_Cis'), (bond.GetProp('Stereochemistry') == 'Bond_Trans')] 
        else:
            s_list = [0, 0]

        return s_list    
        

    n_node = mol.GetNumAtoms()
    n_edge = mol.GetNumBonds() * 2

    D_list, A_list = _DA(mol)
    rings = mol.GetRingInfo().AtomRings()
    atom_fea1 = np.eye(len(atom_list), dtype = bool)[[atom_list.index(a.GetSymbol()) for a in mol.GetAtoms()]]
    atom_fea2 = np.eye(len(charge_list), dtype = bool)[[charge_list.index(a.GetFormalCharge()) for a in mol.GetAtoms()]][:,:-1]
    atom_fea3 = np.eye(len(degree_list), dtype = bool)[[degree_list.index(a.GetDegree()) for a in mol.GetAtoms()]][:,:-1]
    atom_fea4 = np.eye(len(hybridization_list), dtype = bool)[[hybridization_list.index(str(a.GetHybridization())) for a in mol.GetAtoms()]][:,:-2]
    atom_fea5 = np.eye(len(hydrogen_list), dtype = bool)[[hydrogen_list.index(a.GetTotalNumHs(includeNeighbors = True)) for a in mol.GetAtoms()]][:,:-1]
    atom_fea6 = np.eye(len(valence_list), dtype = bool)[[valence_list.index(a.GetTotalValence()) for a in mol.GetAtoms()]][:,:-1]
    atom_fea7 = np.array([[(j in D_list), (j in A_list)] for j in range(mol.GetNumAtoms())], dtype = bool)
    atom_fea8 = np.array([_chirality(a) for a in mol.GetAtoms()], dtype = bool)
    atom_fea9 = np.array([[a.IsInRingSize(s) for s in ringsize_list] for a in mol.GetAtoms()], dtype = bool)
    atom_fea10 = np.array([[a.GetIsAromatic(), a.IsInRing()] for a in mol.GetAtoms()], dtype = bool)
    
    node_attr = np.concatenate([atom_fea1, atom_fea2, atom_fea3, atom_fea4, atom_fea5, atom_fea6, atom_fea7, atom_fea8, atom_fea9, atom_fea10], 1)

    #shift = np.array([atom.GetDoubleProp('shift') for atom in mol.GetAtoms()])
    # mask = np.array([atom.GetBoolProp('mask') for atom in mol.GetAtoms()])

    mol_dict['n_node'].append(n_node)
    mol_dict['n_edge'].append(n_edge)
    mol_dict['node_attr'].append(node_attr)

    #mol_dict['shift'].append(shift)
   # mol_dict['mask'].append(mask)
    mol_dict['smi'].append(Chem.MolToSmiles(mol))
    
    if n_edge > 0:

        bond_fea1 = np.eye(len(bond_list), dtype = bool)[[bond_list.index(str(b.GetBondType())) for b in mol.GetBonds()]]
        bond_fea2 = np.array([_stereochemistry(b) for b in mol.GetBonds()], dtype = bool)
        bond_fea3 = [[b.IsInRing(), b.GetIsConjugated()] for b in mol.GetBonds()]   
        
        edge_attr = np.concatenate([bond_fea1, bond_fea2, bond_fea3], 1)
        edge_attr = np.vstack([edge_attr, edge_attr])
        
        bond_loc = np.array([[b.GetBeginAtomIdx(), b.GetEndAtomIdx()] for b in mol.GetBonds()], dtype = int)
        src = np.hstack([bond_loc[:,0], bond_loc[:,1]])
        dst = np.hstack([bond_loc[:,1], bond_loc[:,0]])
        
        mol_dict['edge_attr'].append(edge_attr)
        mol_dict['src'].append(src)
        mol_dict['dst'].append(dst)
    
    return mol_dict

def  get_mask_shift(mol):
    mask=[]
    shift=[]
    atoms = mol.GetAtoms()
    for atom in atoms:
        num=atom.GetAtomicNum()
        if num==6:
            mask.append(True)
            shift.append(float(1))
        else:
            mask.append(False)
            shift.append(float(0))
    return (mask,shift)

In [21]:
testdata=["CC(C/C=C/CC)N"]

In [22]:
mol_dict = {'n_node': [],
            'n_edge': [],
            'node_attr': [],
            'edge_attr': [],
            'src': [],
            'dst': [],
            'shift': [],
            'mask': [],
            'smi': []}
#先添加化学位移和mask参数
mol_list=[]
for smi in testdata: 
    mol=Chem.MolFromSmiles(smi)
    mask,shift=get_mask_shift(mol)
    mol_dict['mask'].append(mask)
    mol_dict['shift'].append(shift)
    mol_list.append(mol)

#定义基本的参数
atom_list = ['Li','B','C','N','O','F','Na','Mg','Al','Si','P','S','Cl','K','Ti','Zn','Ge','As','Se','Br','Pd','Ag','Sn','Sb','Te','I','Hg','Tl','Pb','Bi']
charge_list = [1, 2, 3, -1, -2, -3, 0]
degree_list = [1, 2, 3, 4, 5, 6, 0]
valence_list = [1, 2, 3, 4, 5, 6, 0]
hybridization_list = ['SP','SP2','SP3','SP3D','SP3D2','S','UNSPECIFIED']
hydrogen_list = [1, 2, 3, 4, 0]
ringsize_list = [3, 4, 5, 6, 7, 8]
bond_list = ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']

rdBase.DisableLog('rdApp.error') 
rdBase.DisableLog('rdApp.warning')
chem_feature_factory = ChemicalFeatures.BuildFeatureFactory(os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef'))

#添加其他参数
for mol in mol_list:
    print(mol)
    Chem.SanitizeMol(mol)
    mol = Chem.RemoveHs(mol)
    mol_dict = add_mol(mol_dict, mol)
    
#格式转换和保存
mol_dict['n_node'] = np.array(mol_dict['n_node']).astype(int)
mol_dict['n_edge'] = np.array(mol_dict['n_edge']).astype(int)
mol_dict['node_attr'] = np.vstack(mol_dict['node_attr']).astype(bool)
mol_dict['edge_attr'] = np.vstack(mol_dict['edge_attr']).astype(bool)
mol_dict['src'] = np.hstack(mol_dict['src']).astype(int)
mol_dict['dst'] = np.hstack(mol_dict['dst']).astype(int)
mol_dict['shift'] = np.hstack(mol_dict['shift'])
mol_dict['mask'] = np.hstack(mol_dict['mask']).astype(bool)
mol_dict['smi'] = np.array(mol_dict['smi'])

for key in mol_dict.keys(): print(key, mol_dict[key].shape, mol_dict[key].dtype)
    
np.savez_compressed('./DataForPrediction.npz', data = [mol_dict])


n_node (1,) int64
n_edge (1,) int64
node_attr (8, 69) bool
edge_attr (14, 8) bool
src (14,) int64
dst (14,) int64
shift (8,) float64
mask (8,) bool
smi (1,) <U13
